# 🎹 Aria — Chopin Style Finetuning
**Goal:** Finetune EleutherAI Aria on Chopin MIDI files so it generates music with Chopin's specific harmonic language, ornamental style, and emotional depth.

**Method:** LoRA (Low-Rank Adaptation) — efficient finetuning that modifies a small number of parameters, preserving the model's piano/musical knowledge while shifting its style toward Chopin.

### What you need
- A folder of **Chopin MIDI files** (80–200 files is enough). Performance MIDIs are better than score MIDIs.
- **Colab A100 GPU** (recommended) or T4 (slower but works)
- This will take **1–3 hours** for a good finetune on T4

### Where to get Chopin MIDIs (free)
- [Kunstderfuge](http://www.kunstderfuge.com/chopin.htm) — large collection of performance MIDIs
- [Mutopia Project](https://www.mutopiaproject.org/cgibin/make-table.cgi?Composer=ChopinFF) — free, clean scores
- [Piano MIDI de](http://www.piano-midi.de/chopin.htm) — high quality performance MIDIs
- [IMSLP](https://imslp.org/) — search "Chopin" → MIDI files section

**Tip:** Mix different genres (nocturnes, mazurkas, waltzes, etudes, preludes) for a more general Chopin style rather than one that only sounds like one piece type.

---

**Runtime:** Use **A100** (Colab Pro) if available. T4 will work but is ~3x slower.

## Step 1 — Install Dependencies

In [ ]:
!pip install -q git+https://github.com/EleutherAI/aria-utils.git
!pip install -q transformers safetensors huggingface_hub peft accelerate
!pip install -q pretty_midi tqdm

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\n✅ Dependencies installed")
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Step 2 — Upload Your Chopin MIDI Dataset

Upload a zip file containing all your Chopin MIDI files, or upload them individually.

In [ ]:
import os, zipfile, glob
from google.colab import files

MIDI_DIR = "chopin_midi"
os.makedirs(MIDI_DIR, exist_ok=True)

print("Upload your Chopin MIDI files (zip or individual .mid files):")
print("→ If you have a zip, upload that. If individual files, upload them all at once.")
print()

uploaded = files.upload()

for fname, data in uploaded.items():
    fpath = os.path.join(MIDI_DIR, fname)
    with open(fpath, 'wb') as f:
        f.write(data)

    # Unzip if it's a zip
    if fname.endswith('.zip'):
        print(f"Extracting {fname}...")
        with zipfile.ZipFile(fpath, 'r') as zf:
            zf.extractall(MIDI_DIR)
        os.remove(fpath)

# Collect all MIDI files recursively
midi_files = glob.glob(f"{MIDI_DIR}/**/*.mid", recursive=True) + \
             glob.glob(f"{MIDI_DIR}/**/*.midi", recursive=True) + \
             glob.glob(f"{MIDI_DIR}/*.mid") + \
             glob.glob(f"{MIDI_DIR}/*.midi")
midi_files = sorted(set(midi_files))

print(f"\n✅ Found {len(midi_files)} MIDI files")
if len(midi_files) < 20:
    print("⚠️  Less than 20 files — the model can still finetune but style transfer will be limited.")
    print("   Recommended: 80+ files for a recognizable Chopin style.")
elif len(midi_files) >= 80:
    print("👍 Great dataset size! This should produce a strong Chopin style.")

## Step 3 — Load the Base Model + Tokenizer

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file

print("Loading Aria model (downloading ~2.8 GB on first run)...")

model = AutoModelForCausalLM.from_pretrained(
    "loubb/aria-medium-base",
    trust_remote_code=True,
    torch_dtype=torch.float16,
)

# Load the generation-finetuned checkpoint as our starting point
# (better to finetune from gen checkpoint than from raw base)
print("Loading generation checkpoint as starting point...")
gen_ckpt_path = hf_hub_download(
    repo_id="loubb/aria-medium-base",
    filename="model-gen.safetensors"
)
gen_state_dict = load_file(gen_ckpt_path)
model.load_state_dict(gen_state_dict, strict=False)

tokenizer = AutoTokenizer.from_pretrained(
    "loubb/aria-medium-base",
    trust_remote_code=True,
)

print("✅ Base model loaded")

## Step 4 — Apply LoRA for Efficient Finetuning

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

# ╔══════════════════════════════════════════╗
# ║           LORA CONFIGURATION            ║
# ╚══════════════════════════════════════════╝

LORA_R          = 32    # LoRA rank. Higher = more capacity but more VRAM
                        #   16 = light finetuning (safe for T4)
                        #   32 = balanced (recommended, works on T4+A100)
                        #   64 = heavy finetuning (A100 recommended)

LORA_ALPHA      = 64    # Usually 2x LORA_R is a good default

LORA_DROPOUT    = 0.05  # Small dropout to avoid overfitting on a small dataset

# Target the attention layers — these control musical style and pattern recognition
# q_proj and v_proj give the best style transfer for music
TARGET_MODULES  = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj"]

# ══════════════════════════════════════════

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Move to device after LoRA wrapping
model = model.to(DEVICE)
print("\n✅ LoRA applied — model is ready to finetune")

## Step 5 — Prepare the Dataset

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import random

# ╔══════════════════════════════════════════╗
# ║         DATASET CONFIGURATION           ║
# ╚══════════════════════════════════════════╝

SEQ_LEN       = 2048   # Sequence length for training. 2048 = ~1 min of music
                       #   Longer = better long-range structure but more VRAM
                       #   T4 can handle 2048 with batch_size=2

STRIDE        = 512    # How much we shift the window when chopping long MIDIs
                       #   Smaller = more training examples from same file

# ══════════════════════════════════════════

class ChopinMIDIDataset(Dataset):
    def __init__(self, midi_files, tokenizer, seq_len, stride):
        self.samples = []
        failed = 0

        print(f"Tokenizing {len(midi_files)} MIDI files...")
        for path in tqdm(midi_files):
            try:
                tokens = tokenizer.encode_from_file(path)
                if tokens is None or len(tokens) < 64:
                    failed += 1
                    continue
                token_ids = tokens if isinstance(tokens, list) else tokens.tolist()

                # Chop into overlapping windows of SEQ_LEN
                for start in range(0, max(1, len(token_ids) - seq_len), stride):
                    chunk = token_ids[start : start + seq_len]
                    if len(chunk) < 64:  # Skip very short chunks
                        continue
                    # Pad to SEQ_LEN if needed
                    if len(chunk) < seq_len:
                        pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id else 0
                        chunk = chunk + [pad_id] * (seq_len - len(chunk))
                    self.samples.append(torch.tensor(chunk, dtype=torch.long))
            except Exception as e:
                failed += 1

        random.shuffle(self.samples)
        print(f"\n✅ Dataset ready: {len(self.samples)} training sequences")
        print(f"   From {len(midi_files) - failed} valid files ({failed} failed to parse)")
        print(f"   Approx. total music: ~{len(self.samples) * seq_len * 0.03 / 60:.0f} min")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        ids = self.samples[idx]
        return {"input_ids": ids, "labels": ids.clone()}


dataset = ChopinMIDIDataset(midi_files, tokenizer, SEQ_LEN, STRIDE)

if len(dataset) == 0:
    raise ValueError("No training samples were created! Check that your MIDI files are valid.")

## Step 6 — Train! 🎼

Adjust training settings below. For a first run, the defaults are safe.

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

# ╔══════════════════════════════════════════╗
# ║        TRAINING CONFIGURATION           ║
# ╚══════════════════════════════════════════╝

NUM_EPOCHS       = 3      # Epochs to train.
                          #   2–4 is usually enough for a strong style shift
                          #   >6 risks overfitting (model only generates Chopin verbatim)

BATCH_SIZE       = 2      # Per-device batch size.
                          #   T4 (16GB): use 2 with SEQ_LEN=2048
                          #   A100 (40GB): use 4–8 with SEQ_LEN=2048

GRAD_ACCUM       = 4      # Effective batch = BATCH_SIZE × GRAD_ACCUM
                          #   Keep effective batch ~8–16 for stable training

LEARNING_RATE    = 2e-4   # LoRA learning rate. 1e-4 to 3e-4 works well.

SAVE_STEPS       = 100    # Save a checkpoint every N steps

OUTPUT_DIR       = "aria_chopin_lora"  # Where checkpoints are saved

# ══════════════════════════════════════════

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    fp16=True,
    logging_steps=10,
    save_steps=SAVE_STEPS,
    save_total_limit=3,        # Keep only the 3 most recent checkpoints
    report_to="none",          # Disable wandb/tensorboard
    dataloader_pin_memory=True,
    remove_unused_columns=False,
)

# Data collator — pads/batches sequences for causal LM
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,  # Causal LM, not masked LM
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    data_collator=data_collator,
)

# Estimate training time
steps_per_epoch = len(dataset) // (BATCH_SIZE * GRAD_ACCUM)
total_steps = steps_per_epoch * NUM_EPOCHS
print(f"Training plan:")
print(f"  Dataset size    : {len(dataset)} sequences")
print(f"  Steps per epoch : ~{steps_per_epoch}")
print(f"  Total steps     : ~{total_steps}")
print(f"  Effective batch : {BATCH_SIZE * GRAD_ACCUM}")
print(f"\n  ⏱ Estimated time on T4  : ~{total_steps * 4 / 60:.0f}–{total_steps * 6 / 60:.0f} min")
print(f"  ⏱ Estimated time on A100: ~{total_steps * 1.5 / 60:.0f}–{total_steps * 2.5 / 60:.0f} min")
print()
print("Starting training...")
print("Watch the loss — it should drop from ~5–7 down to ~2–3 for a good finetune.")
print()

trainer.train()

## Step 7 — Save the LoRA Weights

In [ ]:
LORA_SAVE_DIR = "aria_chopin_lora_final"
model.save_pretrained(LORA_SAVE_DIR)
tokenizer.save_pretrained(LORA_SAVE_DIR)

print(f"✅ LoRA weights saved to '{LORA_SAVE_DIR}/'")
print()
print("Files saved:")
for f in os.listdir(LORA_SAVE_DIR):
    size = os.path.getsize(os.path.join(LORA_SAVE_DIR, f)) / 1e6
    print(f"  {f} ({size:.1f} MB)")
print()
print("💡 These are just the LoRA ADAPTER weights — much smaller than the full model!")
print("   To use them, you load the base model first, then apply these weights on top.")

## Step 8 — Test Your Chopin Model! 🎹

Let's immediately generate something with the freshly finetuned model.

In [ ]:
import os

GEN_OUTPUT_DIR = "chopin_finetune_output"
os.makedirs(GEN_OUTPUT_DIR, exist_ok=True)

# ╔══════════════════════════════════════════╗
# ║      POST-FINETUNE GENERATION           ║
# ╚══════════════════════════════════════════╝

SEED_FILE          = midi_files[0]   # Use the first Chopin file in your dataset as seed
SEED_DURATION_SEC  = 12              # How much of seed to use (seconds)
MAX_NEW_TOKENS     = 3500            # ~2 min of music
TEMPERATURE        = 0.97
TOP_P              = 0.95
NUM_VARIATIONS     = 2

# ══════════════════════════════════════════

model.eval()

print(f"Seed file: {os.path.basename(SEED_FILE)}")
print(f"Generating {NUM_VARIATIONS} Chopin-style variations...\n")

for i in range(NUM_VARIATIONS):
    print(f"  Generating variation {i+1}/{NUM_VARIATIONS}...", end="", flush=True)

    prompt = tokenizer.encode_from_file(
        SEED_FILE,
        return_tensors="pt",
        truncate_seconds=SEED_DURATION_SEC,
    )
    input_ids = prompt.input_ids.to(DEVICE)

    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    midi_dict = tokenizer.decode(output[0].tolist())
    out_path  = os.path.join(GEN_OUTPUT_DIR, f"chopin_ft_variation_{i+1}.mid")
    midi_dict.to_midi().save(out_path)
    print(f" ✅ → {out_path}")

print(f"\n🎉 Done!")

In [ ]:
# Listen to the results
!apt-get install -qq fluidsynth
!pip install -q midi2audio
!wget -q -O /usr/share/sounds/sf2/MuseScore_General.sf2 \
    "https://ftp.osuosl.org/pub/musescore/soundfont/MuseScore_General.sf2"

from midi2audio import FluidSynth
from IPython.display import Audio, display

SF2_PATH = "/usr/share/sounds/sf2/MuseScore_General.sf2"
fs = FluidSynth(sound_font=SF2_PATH)

for midi_path in sorted(glob.glob(f"{GEN_OUTPUT_DIR}/*.mid")):
    wav_path = midi_path.replace(".mid", ".wav")
    fs.midi_to_audio(midi_path, wav_path)
    name = os.path.basename(midi_path)
    print(f"\n🎹 {name}")
    display(Audio(wav_path))

## Step 9 — Download Everything

In [ ]:
import zipfile
from google.colab import files

zip_path = "aria_chopin_finetune_package.zip"
with zipfile.ZipFile(zip_path, 'w') as zf:
    # Include the LoRA weights
    for f in os.listdir(LORA_SAVE_DIR):
        zf.write(os.path.join(LORA_SAVE_DIR, f), arcname=f"lora_weights/{f}")
    # Include generated MIDIs
    for midi_path in glob.glob(f"{GEN_OUTPUT_DIR}/*.mid"):
        zf.write(midi_path, arcname=f"generated/{os.path.basename(midi_path)}")
    # Include WAVs if they exist
    for wav_path in glob.glob(f"{GEN_OUTPUT_DIR}/*.wav"):
        zf.write(wav_path, arcname=f"generated/{os.path.basename(wav_path)}")

print(f"Downloading {zip_path}...")
files.download(zip_path)

---
## 📋 How to Reuse the LoRA Weights Later

Once you have the LoRA weights downloaded, you can load your Chopin model in any notebook:

```python
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# Load base model
base_model = AutoModelForCausalLM.from_pretrained(
    "loubb/aria-medium-base", trust_remote_code=True
)

# Apply your Chopin LoRA
model = PeftModel.from_pretrained(base_model, "./aria_chopin_lora_final")
model = model.merge_and_unload()  # Merge for faster inference
```

---
## 💡 Improving the Results

**If the output still sounds too generic (not Chopin enough):**
- Train for more epochs (try 5–6)
- Increase `LORA_R` to 64
- Add more Chopin MIDIs, especially performance ones
- Use only one genre of Chopin for more focused style (e.g., only nocturnes)

**If the output sounds like it's copying specific pieces (overfitting):**
- Reduce epochs
- Add more variety to your dataset
- Increase `LORA_DROPOUT` to 0.1
- Try using the `example-prompts` from Aria as seeds instead of Chopin pieces

**For longer/better generation:**
- Increase `MAX_NEW_TOKENS` up to `6000–8000` (longer but slower)
- Try `TEMPERATURE=1.0`, `TOP_P=0.93` for a slightly different balance
- Use a Chopin piece from a different composer type as seed for variety

**For best sound quality when rendering:**
- Use [Pianoteq](https://www.modartt.com/) (paid) for a realistic piano sound instead of FluidSynth
- Or use a high-quality Salamander Grand Piano soundfont